# BLAST

![BLAST](https://proto-bio.github.io/proto-assets/images/tool/blast/hero.png)

This notebook demonstrates the two BLAST tools: `run_create_blast_db` builds a local BLAST database from a FASTA file, and `run_blast_search` searches sequences against either a local database or NCBI's online servers. BLAST+ binaries are downloaded automatically on first use, so no manual installation is required.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("blast")
display_overview("blast")
display_docs_section("blast", "Background")

# BLAST

BLAST (Basic Local Alignment Search Tool) finds regions of similarity between biological sequences. It compares nucleotide or protein sequences to sequence databases and calculates the statistical significance of matches. This module provides a unified interface for both *Online BLAST* (querying NCBI servers remotely) and *Local BLAST+* (running searches against custom or downloaded databases on your own hardware), as well as utilities for creating custom BLAST databases.

## Background

**What does this tool do?**
[BLAST](https://blast.ncbi.nlm.nih.gov/Blast.cgi) finds regions of local similarity between sequences by comparing nucleotide or protein sequences to sequence databases like [NCBI GenBank](https://www.ncbi.nlm.nih.gov/genbank/) and calculates the statistical significance of matches.

**Why is this important?**
Sequence alignment is the first step in almost all bioinformatics workflows. It allows researchers to:
- Infer functional relationships: If a new sequence resembles a known gene, they likely share a function.
- Identify species: Map unknown DNA reads to specific organisms (metagenomics).
- Detect off-targets: Ensure a primer or CRISPR guide only binds to the intended target.
- Find evolutionary origins: Trace the phylogeny of a gene across species.

**Scientific foundation:**
BLAST uses a heuristic algorithm that seeks high-scoring segment pairs (HSPs). It does not perform a full [Smith-Waterman](https://en.wikipedia.org/wiki/Smith%E2%80%93Waterman_algorithm) alignment (which is accurate but slow). Instead, it does:

1. **Seeding**: Breaks the query into short "words" (k-mers) and finds exact matches in the database.
2. **Extension**: Extends these matches in both directions until the alignment score drops below a threshold.
3. **Evaluation**: Calculates an [E-value](https://en.wikipedia.org/wiki/BLAST_(biotechnology)#Algorithm) (Expect value) based on the Karlin-Altschul statistics, representing the number of hits one can expect to see by chance.

## Available tools

In [2]:
display_available_tools("blast")

- **`run_create_blast_db()`** — Create a local BLAST database from a FASTA file
- **`run_blast_search()`** — Search sequences against BLAST databases (online or local)

### `run_create_blast_db`

`run_create_blast_db` builds a local BLAST database from a FASTA file using the `makeblastdb` utility from BLAST+. It produces a set of indexed database files whose base path can then be passed directly to `BlastSearchConfig.local_db`. Both nucleotide and protein databases are supported via the `dbtype` parameter.

In [3]:
import tempfile
from pathlib import Path

from proto_tools import (
    CreateBlastDbInput,
    CreateBlastDbConfig,
    run_create_blast_db,
)

In [4]:
# Display input docs
display_api_reference("blast", "input", "run_create_blast_db")

# Write a small FASTA with three well-characterized human globin proteins
tmp_dir = tempfile.mkdtemp(prefix="blast_example_")

fasta_path = Path(tmp_dir) / "test_sequences.fasta"
fasta_path.write_text(
    ">sp|P69905|HBA_HUMAN Hemoglobin subunit alpha\n"
    "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH\n"
    "GSAQVKGHGKKVADALTNAVAHVDDMPNALSALSDLHAHKLRVDPVNFKL\n"
    "LSHCLLVTLAAHLPAEFTPAVHASLDKFLASVSTVLTSKYR\n"
    ">sp|P68871|HBB_HUMAN Hemoglobin subunit beta\n"
    "MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLST\n"
    "PDAVMGNPKVKAHGKKVLGAFSDGLAHLDNLKGTFATLSELHCDKLHVDP\n"
    "ENFRLLGNVLVCVLAHHFGKEFTPPVQAAYQKVVAGVANALAHKYH\n"
    ">sp|P02144|MYG_HUMAN Myoglobin\n"
    "MGLSDGEWQLVLNVWGKVEADIPGHGQEVLIRLFKGHPETLEKFDKFKHL\n"
    "KSEDEMKASEDLKKHGATVLTALGGILKKKGHHEAEIKPLAQSHATKHKIP\n"
    "VKYLEFISECIIQVLQSKHPGDFGADAQGAMNKALELFRKDMASNYKELGFQG\n"
)
print(f"Wrote test FASTA to: {fasta_path}")

db_input = CreateBlastDbInput(fasta=str(fasta_path))

**Input** — `CreateBlastDbInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `fasta` | `str` | required | Path to the input FASTA file containing sequences to create a BLAST database from |

Wrote test FASTA to: /tmp/blast_example_yvfhybil/test_sequences.fasta


In [5]:
# Display config docs
display_api_reference("blast", "config", "run_create_blast_db")

# Create a protein database with a descriptive title | see docs above for all fields
db_config = CreateBlastDbConfig(dbtype="prot", title="Test Protein DB")

**Config** — `CreateBlastDbConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cpu'` | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| `timeout` | `int` | `600` | Maximum execution time in seconds |
| `seed` | `int \| None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `dbtype` | `Literal['nucl', 'prot']` | `'nucl'` | Database type: `nucl` for DNA/RNA, `prot` for amino acid. Must match the input FASTA. |
| `out_prefix` | `str \| None` | `None` | File-path prefix for the generated DB files; falls back to the input FASTA stem when None. |
| `title` | `str \| None` | `None` | Descriptive DB title shown in search reports; `makeblastdb` falls back to the input file name. |
| `parse_seqids` | `bool` | `False` | Parse FASTA seq IDs so `blastdbcmd` can address sequences by ID; required for v5 taxonomy. |
| `hash_index` | `bool` | `False` | Build a hash index of sequence IDs for faster ID lookups; usually paired with `parse_seqids`. |
| `blastdb_version` | `Literal[4, 5]` | `5` | BLAST DB format version: `5` (taxonomy-aware, default since BLAST+ 2.10) or `4` (legacy). |
| `max_file_sz` | `str` | `'1GB'` | Max size per DB volume with unit suffix (e.g. `1GB`, `500MB`); `4GB` is the upstream max. |
| `taxid` | `int \| None` | `None` | NCBI taxonomy ID assigned to every sequence; set to tag a single-organism DB. |
| `extra_args` | `list[str]` | `[]` | Verbatim `makeblastdb` flags for fields not exposed above (e.g. `-mask_data /path`). |

In [6]:
# Run database creation
db_result = run_create_blast_db(db_input, db_config)

Running run_create_blast_db [00:00]

In [7]:
# Display output docs
display_api_reference("blast", "output", "run_create_blast_db")

# The db_path is passed directly to BlastSearchConfig.local_db for searches
print(f"Database created at: {db_result.db_path}")

**Output** — `CreateBlastDbOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `db_path` | `str` | required | Path to the generated BLAST database |

Database created at: /tmp/blast_example_yvfhybil/test_sequences


### `run_blast_search`

`run_blast_search` performs sequence similarity search in either local or online mode. Local mode runs BLAST+ against a database created by `run_create_blast_db`, while online mode submits the query to NCBI's QBLAST service and returns hits from large public databases such as SwissProt or nr. The query accepts either a raw sequence string or a path to a FASTA file, and results are returned as a list of `BlastHit` objects with the standard 12-column tabular fields.

In [8]:
from proto_tools import (
    BlastSearchInput,
    BlastSearchConfig,
    run_blast_search,
)

In [9]:
# Display input docs
display_api_reference("blast", "input", "run_blast_search")

# Write a FASTA query file with the hemoglobin beta sequence
query_seq = "MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLSTPDAVMGNPKVKAHGKKVLGAFSDGLAHLDNLKGTFATLSELHCDKLHVDPENFRLLGNVLVCVLAHHFGKEFTPPVQAAYQKVVAGVANALAHKYH"

query_path = Path(tmp_dir) / "query.fasta"
query_path.write_text(f">query_hbb\n{query_seq}\n")

search_input = BlastSearchInput(query=str(query_path))

**Input** — `BlastSearchInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `query` | `str` | required | Query sequence string or path to a FASTA file containing query sequence(s) |
| `query_type` | `Literal['sequence', 'fasta_path']` | `'sequence'` | Auto-inferred query type: 'sequence' for raw string, 'fasta_path' for a file path. |

In [10]:
# Display config docs
display_api_reference("blast", "config", "run_blast_search")

# Local blastp search against the database we just built | see docs above for all fields
search_config = BlastSearchConfig(
    search_mode="local",
    program="blastp",
    local_db=db_result.db_path,
    num_threads=2,
)

**Config** — `BlastSearchConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cpu'` | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| `timeout` | `int` | `600` | Maximum execution time in seconds |
| `seed` | `int \| None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `search_mode` | `Literal['online', 'local']` | `'online'` | `online` queries NCBI's QBLAST servers; `local` runs BLAST+ CLI against a local database. |
| `program` | `Literal['blastn', 'blastp', 'blastx', 'tblastn', 'tblastx']` | `'blastn'` | BLAST algorithm (blastn/blastp/blastx/tblastn/tblastx). |
| `database` | `Literal['nt', 'nr', 'refseq_rna', 'refseq_protein', 'swissprot', 'pdb', 'pataa', 'patnt']` | `'nt'` | NCBI database to search against (e.g. `nt`, `nr`, `swissprot`, `pdb`). |
| `entrez_query` | `str \| None` | `None` | Restrict the online search with an Entrez query (e.g. `Homo sapiens[Organism]`). |
| `hitlist_size` | `int \| None` | `None` | Number of database sequences to return; NCBI defaults to 50 when unset. |
| `megablast` | `bool \| None` | `None` | Use the MegaBLAST algorithm for `blastn` (faster, optimized for highly similar sequences). |
| `local_db` | `str \| None` | `None` | Path to a local BLAST database stem (no file extensions). Required for local mode. |
| `num_threads` | `int` | `4` | CPU threads for local BLAST search (upstream BLAST+ default is 1). |
| `evalue` | `float \| None` | `None` | Expectation value threshold for reporting hits; BLAST+ default is 10.0 when unset. |
| `word_size` | `int \| None` | `None` | Length of the initial exact match; BLAST+ defaults: 28 (megablast), 11 (blastn), 3 (blastp). |
| `gapopen` | `int \| None` | `None` | Cost to open a gap; BLAST+ defaults: 5 (blastn), 11 (protein programs). |
| `gapextend` | `int \| None` | `None` | Cost to extend a gap; BLAST+ defaults 2 (blastn), 1 (protein). Not for tblastx. |
| `matrix` | `Literal['BLOSUM45', 'BLOSUM50', 'BLOSUM62', 'BLOSUM80', 'BLOSUM90', 'PAM30', 'PAM70', 'PAM250'] \| None` | `None` | Substitution matrix for protein alignments; BLAST+ default is BLOSUM62. Not applicable to blastn. |
| `reward` | `int \| None` | `None` | Match reward (blastn only); BLAST+ defaults 1 (megablast), 2 (blastn). |
| `penalty` | `int \| None` | `None` | Mismatch penalty (blastn, ≤0); BLAST+ defaults -2 (megablast), -3 (others). |
| `threshold` | `int \| None` | `None` | Minimum word score for the BLAST lookup table (protein only); BLAST+ default is 11 for blastp. |
| `comp_based_stats` | `Literal[0, 1, 2, 3] \| None` | `None` | Composition-based scoring (protein): 0=off, 1=stats, 2=adjust (default), 3=unconditional. |
| `max_target_seqs` | `int \| None` | `None` | Maximum number of aligned sequences to keep; BLAST+ default is 500. |
| `perc_identity` | `float \| None` | `None` | Minimum percent identity for reported alignments (0-100); most useful with blastn. |
| `qcov_hsp_perc` | `float \| None` | `None` | Minimum query coverage percentage per HSP (0-100). |
| `soft_masking` | `bool \| None` | `None` | Apply filter as soft masks (seeding only); BLAST+ defaults True (blastn), False (protein). |
| `lcase_masking` | `bool \| None` | `None` | Treat lowercase letters in the FASTA input as masked. |
| `dust` | `str \| None` | `None` | Nucleotide low-complexity filter: `yes`, `no`, or `level window linker` (default `20 64 1`). |
| `seg` | `str \| None` | `None` | Protein low-complexity filter: `yes`, `no`, or `window locut hicut` (blastp default `no`). |
| `task` | `Literal['megablast', 'dc-megablast', 'blastn', 'blastn-short', 'blastp', 'blastp-short', 'blastp-fast', 'blastx', 'blastx-fast', 'tblastn', 'tblastn-fast'] \| None` | `None` | Task preset (e.g. `megablast`, `blastp-fast`, `blastx-fast`); flips multiple defaults at once. |
| `ungapped` | `bool \| None` | `None` | Perform ungapped alignment only. |
| `strand` | `Literal['both', 'plus', 'minus'] \| None` | `None` | Query strand(s) to search; BLAST+ default is `both`. Only for blastn, blastx, tblastx. |
| `query_gencode` | `int \| None` | `None` | Genetic code for translating the query (blastx/tblastx only); BLAST+ default 1 (Standard). |
| `db_gencode` | `int \| None` | `None` | Genetic code for translating database sequences (tblastn/tblastx only); BLAST+ default 1. |
| `extra_args` | `list[str]` | `[]` | Verbatim BLAST+ CLI tokens for niche flags (e.g. `['-max_hsps', '1']`). Local mode only. |

In [11]:
# Run the local BLAST search
result = run_blast_search(search_input, search_config)

Running run_blast_search [00:00]

In [12]:
# Display output docs
display_api_reference("blast", "output", "run_blast_search")

# HBB should self-hit at 100% identity; HBA should appear at ~43% identity
print(f"Found {result.num_hits} hits\n")
for hit in result.hits:
    print(f"  {hit.sseqid}: {hit.pident:.1f}% identity, e-value={hit.evalue:.1e}, bitscore={hit.bitscore:.1f}")

**Output** — `BlastSearchOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `hits` | `list[proto_tools.tools.sequence_alignment.blast.blast_search.BlastHit]` | `[]` | BLAST alignment hits |

Found 2 hits

  sp|P68871|HBB_HUMAN: 100.0% identity, e-value=1.4e-111, bitscore=301.0
  sp|P69905|HBA_HUMAN: 43.4% identity, e-value=5.4e-38, bitscore=114.0


#### Online search

For searches against large public databases, set `search_mode="online"`. The query is submitted to NCBI's QBLAST service and requires an internet connection. The raw sequence string can be passed directly to `BlastSearchInput` without writing a FASTA file.

In [13]:
# Online search accepts a raw sequence string directly
online_input = BlastSearchInput(query=query_seq)
online_config = BlastSearchConfig(
    search_mode="online",
    program="blastp",
    database="swissprot",
    hitlist_size=5,
)

online_result = run_blast_search(online_input, online_config)
print(f"Found {online_result.num_hits} hits\n")
for hit in online_result.hits:
    print(f"  {hit.sseqid}: {hit.pident:.1f}% identity, e-value={hit.evalue:.1e}, bitscore={hit.bitscore:.1f}")

Found 5 hits

  P68871: 100.0% identity, e-value=5.8e-106, bitscore=301.2
  P02024: 99.3% identity, e-value=2.0e-105, bitscore=300.1
  P02025: 98.6% identity, e-value=3.0e-103, bitscore=294.3
  P02032: 97.3% identity, e-value=2.4e-102, bitscore=292.0
  P19885: 95.9% identity, e-value=4.3e-102, bitscore=291.6


## Export Results

Search results can be exported to CSV or JSON for downstream analysis. CSV produces one row per hit with the standard 12 tabular fields; JSON preserves the same structure for programmatic consumption.

In [14]:
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export(name="blast_search_local_results", export_path=output_dir, file_format="csv")
online_result.export(name="blast_search_online_results", export_path=output_dir, file_format="csv")
print(f"Results exported to {output_dir}")

Results exported to example_output
